In [3]:
from operator import itemgetter

from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END
from typing import TypedDict
from flashrank import Ranker, RerankRequest
import psycopg2

In [5]:
conn = psycopg2.connect("dbname=rag_db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id SERIAL PRIMARY KEY,
    name TEXT,
    course TEXT,
    grade REAL,
    semester TEXT
);

DELETE FROM students;

INSERT INTO students (name, course, grade, semester) VALUES
('Alice', 'Machine Learning', 92.5, 'Fall 2025'),
('Bob', 'Machine Learning', 78.0, 'Fall 2025'),
('Carol', 'Deep Learning', 88.3, 'Fall 2025'),
('Dave', 'Deep Learning', 95.1, 'Fall 2025'),
('Eve', 'NLP', 67.0, 'Spring 2025'),
('Frank', 'NLP', 81.4, 'Spring 2025'),
('Grace', 'Computer Vision', 90.2, 'Fall 2025'),
('Hank', 'Computer Vision', 73.8, 'Spring 2025');
""")

conn.commit()
conn.close()
print("Sample data loaded into rag_db")

Sample data loaded into rag_db


In [6]:
DATA_DIR = "./RAG_data"

txt_loader = DirectoryLoader(DATA_DIR, glob="./*.txt", loader_cls=TextLoader)
pdf_loader = DirectoryLoader(DATA_DIR, glob="./*.pdf", loader_cls=PyPDFLoader)
docs = txt_loader.load() + pdf_loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
splits = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} documents → {len(splits)} chunks")

Loaded 73 documents → 302 chunks


In [7]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 5

def hybrid_retrieve(query: str) -> list:
    vec_docs = vector_retriever.invoke(query)
    bm25_docs = bm25_retriever.invoke(query)
    seen = {}
    for d in vec_docs + bm25_docs:
        if d.page_content not in seen:
            seen[d.page_content] = d
    return list(seen.values())

print("Hybrid retriever ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hybrid retriever ready


In [8]:
llm = ChatOllama(model="llama3.2", temperature=0)
db = SQLDatabase.from_uri("postgresql://localhost/rag_db")
ranker = Ranker(model_name="ms-marco-MultiBERT-L-12")

CONTEXTUALIZE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, rewrite the question "
     "to be fully self-contained. Do NOT answer it. Only reformulate if needed, "
     "otherwise return it as-is."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])
contextualize_chain = CONTEXTUALIZE_PROMPT | llm | StrOutputParser()

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise research assistant. Answer ONLY from the provided context.\n"
     "If the answer is not in the context, say 'I don't have enough information.'\n"
     "Cite the source for every claim. Be concise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

expansion_prompt = ChatPromptTemplate.from_template(
    "Generate 3 short search queries related to: {question}\nOutput only the queries, one per line."
)
expansion_chain = expansion_prompt | llm | StrOutputParser()

print("LLM, prompts, SQL, reranker ready")

LLM, prompts, SQL, reranker ready


In [9]:
class RAGState(TypedDict):
    question: str
    chat_history: list
    standalone_question: str
    queries: list[str]
    documents: list
    context: str
    answer: str
    retry_count: int
    route: str


def contextualize(state: RAGState) -> dict:
    if state["chat_history"]:
        standalone = contextualize_chain.invoke({
            "question": state["question"],
            "chat_history": state["chat_history"],
        })
    else:
        standalone = state["question"]
    return {"standalone_question": standalone}


def route_question(state: RAGState) -> dict:
    decision = llm.invoke(
        f"Does this question need a SQL database query or a document search?\n"
        f"The database has these tables: {db.get_table_info()}\n"
        f"Question: {state['standalone_question']}\n"
        f"Reply ONLY 'sql' or 'docs'."
    ).content.strip().lower()
    route = "sql" if "sql" in decision else "docs"
    print(f"  → Routed to: {route}")
    return {"route": route}


def query_sql(state: RAGState) -> dict:
    sql = llm.invoke(
        f"Write a PostgreSQL query for this question. Return ONLY the SQL, nothing else.\n"
        f"Tables: {db.get_table_info()}\n"
        f"Question: {state['standalone_question']}"
    ).content.strip().strip("`").replace("sql\n", "")
    print(f"  → SQL: {sql}")
    try:
        result = db.run(sql)
        context = f"[Source: SQL Database]\nQuery: {sql}\nResult: {result}"
    except Exception as e:
        context = f"[Source: SQL Database]\nQuery failed: {sql}\nError: {e}"
    return {"context": context}


def expand_query(state: RAGState) -> dict:
    expanded = expansion_chain.invoke({"question": state["standalone_question"]})
    queries = [state["standalone_question"]] + [
        q.strip() for q in expanded.strip().split("\n") if q.strip()
    ]
    return {"queries": queries}


def retrieve(state: RAGState) -> dict:
    all_docs = []
    for q in state["queries"]:
        all_docs.extend(hybrid_retrieve(q))
    unique = list({d.page_content: d for d in all_docs}.values())
    retries = state.get("retry_count", 0)
    return {"documents": unique, "retry_count": retries}


def rerank_docs(state: RAGState) -> dict:
    docs = state["documents"]
    if not docs:
        return {"context": "No relevant documents found."}
    passages = [{"id": i, "text": d.page_content, "meta": d.metadata} for i, d in enumerate(docs)]
    results = ranker.rerank(RerankRequest(query=state["standalone_question"], passages=passages))
    top_docs = [Document(page_content=r["text"], metadata=r["meta"]) for r in results[:4]]
    context = "\n\n".join(
        f"[Source: {d.metadata.get('source', 'Unknown')}]\n{d.page_content}" for d in top_docs
    )
    return {"context": context}


def check_quality(state: RAGState) -> dict:
    retries = state.get("retry_count", 0)
    if retries >= 2:
        return {}
    check = llm.invoke(
        f"Does this context contain enough information to answer the question?\n"
        f"Question: {state['standalone_question']}\n"
        f"Context: {state['context'][:500]}\n"
        f"Reply ONLY 'yes' or 'no'."
    ).content.strip().lower()
    if "no" in check:
        broader = expansion_chain.invoke({
            "question": f"Broaden this search, find related concepts: {state['standalone_question']}"
        })
        new_queries = [q.strip() for q in broader.strip().split("\n") if q.strip()]
        print(f"  ↻ Retry {retries + 1}: context insufficient, broadening...")
        return {"queries": new_queries, "retry_count": retries + 1}
    return {}


def generate(state: RAGState) -> dict:
    chain = ANSWER_PROMPT | llm | StrOutputParser()
    answer = chain.invoke({
        "context": state["context"],
        "question": state["question"],
        "chat_history": state["chat_history"],
    })
    return {"answer": answer}

In [10]:
graph = StateGraph(RAGState)

graph.add_node("contextualize", contextualize)
graph.add_node("route_question", route_question)
graph.add_node("query_sql", query_sql)
graph.add_node("expand_query", expand_query)
graph.add_node("retrieve", retrieve)
graph.add_node("rerank", rerank_docs)
graph.add_node("check_quality", check_quality)
graph.add_node("generate", generate)

graph.set_entry_point("contextualize")
graph.add_edge("contextualize", "route_question")

graph.add_conditional_edges(
    "route_question",
    lambda state: state.get("route", "docs"),
    {"sql": "query_sql", "docs": "expand_query"},
)

graph.add_edge("query_sql", "generate")
graph.add_edge("expand_query", "retrieve")
graph.add_edge("retrieve", "rerank")
graph.add_edge("rerank", "check_quality")

graph.add_conditional_edges(
    "check_quality",
    lambda state: "retry" if state.get("retry_count", 0) > 0 and len(state["context"]) < 200 else "done",
    {"retry": "retrieve", "done": "generate"},
)

graph.add_edge("generate", END)

rag_graph = graph.compile()
print("Graph compiled (docs + SQL + retry)")

Graph compiled (docs + SQL + retry)


In [11]:
MAX_HISTORY_TURNS = 5
chat_history = []

def ask(query):
    global chat_history
    trimmed = chat_history[-(MAX_HISTORY_TURNS * 2):]

    result = rag_graph.invoke({
        "question": query,
        "chat_history": trimmed,
        "standalone_question": "",
        "queries": [],
        "documents": [],
        "context": "",
        "answer": "",
        "retry_count": 0,
        "route": "",
    })

    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=result["answer"]))
    return result["answer"]

def reset():
    global chat_history
    chat_history = []
    print("History cleared.")

In [12]:
reset()

# SQL questions
print("=" * 60)
print("Q1: How many students scored above 90?")
print("=" * 60)
print(ask("How many students scored above 90?"))

print("\n" + "=" * 60)
print("Q2: What is the average grade in Deep Learning?")
print("=" * 60)
print(ask("What is the average grade in Deep Learning?"))

# Document questions
print("\n" + "=" * 60)
print("Q3: What are the energy transfer strategies?")
print("=" * 60)
print(ask("What are the energy transfer strategies?"))

print("\n" + "=" * 60)
print("Q4: What is the novelty in using NARS scheduler?")
print("=" * 60)
print(ask("What is the novelty in using NARS scheduler?"))

History cleared.
Q1: How many students scored above 90?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  → Routed to: sql


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  → SQL: SELECT COUNT(*) FROM students WHERE grade > 90;


...

  → Routed to: sql


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  → SQL: SELECT * FROM students WHERE course = 'NARS scheduler';


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


I don't have enough information. The query result only shows the students who took the "NARS scheduler" course, but it does not provide any information about what makes the NARS scheduler novel or unique.


## RAG V3 — LangGraph + SQL + Hybrid Retrieval

**What it does:** A local conversational RAG system that answers questions from both documents and a PostgreSQL database, with automatic routing.

**Stack:** LangChain, LangGraph, Ollama (Llama 3.2), ChromaDB, BM25, FlashRank, PostgreSQL

**Graph flow:**
```
User query → Contextualize → Route → SQL path  → Generate
                                    → Docs path → Expand → Retrieve → Rerank → Quality check ↻ → Generate
```

**Key features:**
- **Auto-routing** — LLM decides if a question needs SQL or document retrieval
- **Query contextualization** — follow-up questions are rewritten as standalone before retrieval
- **Hybrid retrieval** — BM25 (keyword) + ChromaDB (semantic) combined
- **Query expansion** — each question generates 3 related queries for broader recall
- **FlashRank reranking** — cross-encoder scores and filters retrieved chunks
- **Retry loop** — if retrieved context is insufficient, broadens the query and retries (up to 2x)
- **Chat history** — multi-turn conversations, trimmed to last 5 turns